# Setup

In [ ]:
!pip install xarray tiler rioxarray

In [ ]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
# from osgeo import gdal
from tqdm import tqdm

%matplotlib inline

In [ ]:
from transformers import AutoImageProcessor
from tiler import Tiler, Merger

In [ ]:
repo_dir = "lfm"

sys.path.append("lfm")

from lfm.tasks.sem_segmentation.sseg_model import load_dinov3_encoder, DINOSegmentation
from lfm.tasks.sem_segmentation.data_cube_inference import run_datacube_inference, plot_data_cubes

# Config

In [ ]:
# Data paths
INPUT_DIR = "/explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/"

# Where to load dinov3 init weights
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/best_model.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs_meeting"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Number of bands, band filter
NUM_BANDS = 3
BAND_FILTER = [3, 1, 0]

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model

In [ ]:
# Load model from model factory
encoder = load_dinov3_encoder(weights_local_checkpoint, device=device) 
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE,
    num_bands=NUM_BANDS,
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(
    checkpoint_state, strict=False
)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

# Load data

## Load statistics from training to normalize inputs

In [ ]:
print("\n" + "="*60)
print("STEP 1: Loading training dataset statistics.")
print("="*60)

# Load mean and std from a previous training run
MEAN = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_mean.npy")
MEAN = MEAN[BAND_FILTER]
STD = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_std.npy")
STD = STD[BAND_FILTER]
print(MEAN, STD)

# Inference

In [ ]:
# plot some data cubes
plot_data_cubes(INPUT_DIR, 1000, 25, ".")

In [ ]:
# Run inference, create viz of inference
n_images = 3
# images, predictions = run_datacube_inference(
#     model=model,
#     device=device,
#     tiff_dir=INPUT_DIR,
#     mean=MEAN,
#     std=STD,
#     output_path="sem_seg_inf_test_3_19.png",
#     n_images=n_images,
#     model_native_size=TARGET_SIZE[0],
#     tile_overlap=0.25,
#     threshold=0.75,
#     normalize=True,
#     save_inputs_dir=None,
#     use_sliding=True,
# )